# 부산항 요청·지정 내역 비교

대상: `부산_풀컨테이너_전체시설_요청기간내_2022-2025.csv`의 62,225행입니다. 풀컨테이너선이며 요청기간 시작·종료가 모두 2022~2025년 안에 있는 자료입니다.

| 조건 | 정의 | 현재 자료의 확인값 |
|---|---|---:|
| ① | 요청기간 시작 또는 종료가 지정기간과 다름. 지정기간 누락 포함 | 1,004행 |
| ② | 요청시설은 신항 1~5부두이고 지정시설은 그 외 또는 누락 | 75행 |
| ③ | ①과 ②를 동시에 만족 | 22행 |

①의 ‘다름’은 **시작·종료 중 하나라도 다름**을 뜻합니다. ②는 두 조건을 `&`로 연결합니다. 지정시설이 신항 1~5부두 내부의 다른 선석으로 바뀐 경우는 ②에 해당하지 않습니다. 지정값 누락은 변경이 확정된 기록과 구분하여 표시합니다.


In [1]:
from pathlib import Path
import pandas as pd

input_path = Path("부산_풀컨테이너_전체시설_요청기간내_2022-2025.csv")

# 호출부호·시설코드·입항횟수 등의 앞자리 0을 보존합니다.
df = pd.read_csv(input_path, encoding="utf-8-sig", dtype="string")

required_columns = [
    "호출부호", "선박명", "요청기간시작", "요청기간종료",
    "지정기간시작", "지정기간종료", "요청시설명", "지정시설명",
]
missing_columns = set(required_columns) - set(df.columns)
if missing_columns:
    raise ValueError(f"필요한 열이 없습니다: {sorted(missing_columns)}")

# 원본 df는 보존하고 비교용 자료만 별도로 정리합니다.
compare = df[required_columns].fillna("").apply(lambda column: column.str.strip())
print(f"전체 데이터: {len(df):,}행")


전체 데이터: 62,225행


## 1. 세 가지 조건 만들기

누락값은 비교할 때 빈 문자열로 바꿉니다. 원본의 시간 표기는 동일한 `YYYY-MM-DD HH:MM` 형식이므로 문자열로 직접 비교합니다. 원본에 잘못된 연도가 있더라도 임의로 날짜를 수정하거나 누락값으로 변환하지 않습니다.


In [2]:
# ① 시작 또는 종료 중 하나라도 다르면 True입니다.
start_different = compare["요청기간시작"].ne(compare["지정기간시작"])
end_different = compare["요청기간종료"].ne(compare["지정기간종료"])
condition_1 = start_different | end_different

designated_period_missing = (
    compare["지정기간시작"].eq("") | compare["지정기간종료"].eq("")
)

# '신항 1부두'부터 '신항 5부두'까지만 선택합니다.
# '신항 11부두'나 '신항 U-1 정박지'는 포함하지 않습니다.
pattern = r"^신항\s*[1-5]부두(?:\s|$)"
request_in_1_to_5 = compare["요청시설명"].str.contains(pattern, regex=True, na=False)
designated_in_1_to_5 = compare["지정시설명"].str.contains(pattern, regex=True, na=False)
designated_facility_missing = compare["지정시설명"].eq("")

# ② 요청시설은 신항 1~5부두이고, 지정시설은 그 밖(누락 포함)입니다.
condition_2 = request_in_1_to_5 & ~designated_in_1_to_5

# ③ 두 조건을 동시에 만족합니다.
condition_3 = condition_1 & condition_2

result_1 = df.loc[condition_1].copy()
result_1["시작불일치"] = start_different.loc[condition_1]
result_1["종료불일치"] = end_different.loc[condition_1]
result_1["지정기간누락"] = designated_period_missing.loc[condition_1]

result_2 = df.loc[condition_2].copy()
result_2["지정시설누락"] = designated_facility_missing.loc[condition_2]

result_3 = df.loc[condition_3].copy()
result_3["시작불일치"] = start_different.loc[condition_3]
result_3["종료불일치"] = end_different.loc[condition_3]
result_3["지정기간누락"] = designated_period_missing.loc[condition_3]
result_3["지정시설누락"] = designated_facility_missing.loc[condition_3]


## 2. 조건별 행 수와 누락 내역

행 수는 시설사용허가 기록의 수이며, 고유 선박 수가 아닙니다. ③은 ①과 ②의 교집합이므로 세 행 수를 합산하지 않습니다.


In [3]:
summary = pd.DataFrame({
    "조건": ["① 기간 불일치(누락 포함)", "② 신항 1~5부두 → 그 외/누락", "③ 조건 ① & 조건 ②"],
    "행수": [len(result_1), len(result_2), len(result_3)],
})
display(summary)

missing_summary = pd.DataFrame({
    "구분": [
        "① 지정기간 시작·종료가 모두 있고 서로 다름",
        "① 지정기간 시작 또는 종료 누락",
        "② 지정시설이 다른 곳으로 기재됨",
        "② 지정시설 누락",
        "③ 중 지정기간 누락",
        "③ 중 지정시설 누락",
    ],
    "행수": [
        int((condition_1 & ~designated_period_missing).sum()),
        int((condition_1 & designated_period_missing).sum()),
        int((condition_2 & ~designated_facility_missing).sum()),
        int((condition_2 & designated_facility_missing).sum()),
        int((condition_3 & designated_period_missing).sum()),
        int((condition_3 & designated_facility_missing).sum()),
    ],
})
display(missing_summary)


,조건,행수
0,① 기간 불일치(누락 포함),1004
1,② 신항 1~5부두 → 그 외/누락,75
2,③ 조건 ① & 조건 ②,22


,구분,행수
0,① 지정기간 시작·종료가 모두 있고 서로 다름,185
1,① 지정기간 시작 또는 종료 누락,819
2,② 지정시설이 다른 곳으로 기재됨,53
3,② 지정시설 누락,22
4,③ 중 지정기간 누락,22
5,③ 중 지정시설 누락,22


## 3. 조건 ①에 해당하는 기록

`result_1`에 **전체 해당 행과 원본의 모든 열**이 있습니다. 아래에는 주요 열의 앞 20행을 표시합니다.


In [4]:
period_columns = [
    "호출부호", "선박명", "요청기간시작", "요청기간종료",
    "지정기간시작", "지정기간종료", "시작불일치", "종료불일치", "지정기간누락",
]
print(f"조건 ①: {len(result_1):,}행 (앞 20행 표시)")
display(result_1[period_columns].head(20))


조건 ①: 1,004행 (앞 20행 표시)


,호출부호,선박명,요청기간시작,요청기간종료,지정기간시작,지정기간종료,시작불일치,종료불일치,지정기간누락
11,D5TZ4,CMA CGM SAN FRANCISCO,2022-01-01 02:00,2022-01-01 12:00,<NA>,<NA>,True,True,True
260,V7A5227,PRESTIGE,2022-01-07 18:00,2022-01-16 18:00,<NA>,<NA>,True,True,True
373,D5QW9,ATLANTIC EAST,2022-01-10 03:00,2022-01-13 18:00,2022-01-10 05:00,2022-01-13 18:00,True,False,False
379,V7A5051,STAR CHASER,2022-01-10 07:00,2022-01-12 17:00,2022-01-11 07:00,2022-01-12 17:00,True,False,False
720,CQGL,MSC ODESSA,2022-01-18 14:00,2022-01-21 23:00,<NA>,<NA>,True,True,True
764,V7A5228,GOOD PROSPECT,2022-01-19 12:00,2022-01-21 14:00,<NA>,<NA>,True,True,True
847,V7UG4,STAR CLIPPER,2022-01-21 15:00,2022-01-22 08:00,<NA>,<NA>,True,True,True
874,KGMQ,CMA CGM HERODOTE,2022-01-22 10:00,2022-01-22 23:00,<NA>,<NA>,True,True,True
880,V7UG4,STAR CLIPPER,2022-01-22 13:00,2022-01-23 08:00,<NA>,<NA>,True,True,True
927,DSQG5,PEGASUS PRIME,2022-01-23 14:00,2022-01-24 06:00,<NA>,<NA>,True,True,True


## 4. 조건 ②에 해당하는 기록

`result_2`에 전체 해당 행과 모든 원본 열이 있습니다. 시설명에는 선석까지 포함됩니다.


In [5]:
facility_columns = [
    "호출부호", "선박명", "요청시설명", "지정시설명",
    "요청기간시작", "요청기간종료", "지정시설누락",
]
print(f"조건 ②: {len(result_2):,}행 (앞 20행 표시)")
display(result_2[facility_columns].head(20))

# 어떤 요청시설에서 어떤 지정시설로 연결되는지 집계합니다.
# 누락값을 '(누락)'으로 표시하여 집계에서 빠지지 않게 합니다.
facility_pairs = (
    result_2[["요청시설명", "지정시설명"]]
    .fillna("(누락)")
    .groupby(["요청시설명", "지정시설명"], dropna=False)
    .size()
    .reset_index(name="행수")
    .sort_values("행수", ascending=False)
    .reset_index(drop=True)
)
display(facility_pairs)


조건 ②: 75행 (앞 20행 표시)


,호출부호,선박명,요청시설명,지정시설명,요청기간시작,요청기간종료,지정시설누락
876,KGMQ,CMA CGM HERODOTE,신항 5부두 1선석,신항 U-2 정박지,2022-01-22 11:00,2022-01-22 23:00,False
1094,DJBF2,SHANGHAI EXPRESS,신항 3부두 4선석,<NA>,2022-01-27 19:00,2022-01-29 05:00,True
1406,CQGL,MSC ODESSA,신항 1부두 1선석,남외항 N-5박지,2022-02-04 17:00,2022-02-05 14:00,False
2073,KDAB,CMA CGM DAKAR,신항 5부두 4선석,신항 U-1 정박지,2022-02-19 09:00,2022-02-21 23:00,False
2367,V7BC2,BALLENITA,신항 5부두 3선석,신항 U-1 정박지,2022-02-26 17:00,2022-02-27 18:00,False
2421,V7A5100,QINGDAO TRADER,신항 3부두 4선석,신항 U-1 정박지,2022-02-28 00:01,2022-02-28 08:00,False
2860,9V9924,APL SANTIAGO,신항 5부두 1선석,남외항 N-5박지,2022-03-10 06:00,2022-03-10 10:00,False
3683,D5QW3,PACIFIC TIANJIN,신항 4부두 2선석,<NA>,2022-03-30 16:00,2022-03-30 23:00,True
3817,VRNX7,SCO QINGDAO,신항 3부두 3선석,신항 U-1 정박지,2022-04-03 04:00,2022-04-04 12:00,False
4195,V7DG7,SPYROS V,신항 2부두 2선석,남외항 N-5박지,2022-04-12 07:00,2022-04-12 18:00,False


,요청시설명,지정시설명,행수
0,신항 2부두 2선석,(누락),3
1,신항 5부두 1선석,(누락),3
2,신항 2부두 2선석,남외항 N-5박지,2
3,신항 1부두 1선석,(누락),2
4,신항 3부두 1선석,(누락),2
5,신항 3부두 3선석,신선대부두 2선석,2
6,신항 5부두 4선석,신항 U-1 정박지,2
7,신항 5부두 1선석,신항 U-2 정박지,2
8,신항 4부두 3선석,(누락),2
9,신항 4부두 1선석,(누락),2


## 5. 조건 ③에 해당하는 기록 전체

`result_3`은 기간 불일치와 신항 1~5부두 밖 지정(누락 포함)을 동시에 만족한 기록입니다. 아래에서 **22행 전체**를 확인할 수 있습니다.


In [6]:
combined_columns = [
    "호출부호", "선박명", "요청시설명", "지정시설명",
    "요청기간시작", "요청기간종료", "지정기간시작", "지정기간종료",
    "지정기간누락", "지정시설누락",
]
print(f"조건 ③: {len(result_3):,}행 (전체 표시)")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(result_3[combined_columns])


조건 ③: 22행 (전체 표시)


,호출부호,선박명,요청시설명,지정시설명,요청기간시작,요청기간종료,지정기간시작,지정기간종료,지정기간누락,지정시설누락
1094,DJBF2,SHANGHAI EXPRESS,신항 3부두 4선석,<NA>,2022-01-27 19:00,2022-01-29 05:00,<NA>,<NA>,True,True
3683,D5QW3,PACIFIC TIANJIN,신항 4부두 2선석,<NA>,2022-03-30 16:00,2022-03-30 23:00,<NA>,<NA>,True,True
4295,UBHR9,IRIDA,신항 1부두 2선석,<NA>,2022-04-15 01:00,2022-04-15 20:00,<NA>,<NA>,True,True
4668,DSRA2,SINOKOR NIIGATA,신항 1부두 1선석,<NA>,2022-04-24 01:00,2022-04-24 10:00,<NA>,<NA>,True,True
4921,CQCZ,ACTUARIA,신항 3부두 1선석,<NA>,2022-04-30 15:00,2022-05-01 15:00,<NA>,<NA>,True,True
4937,5LDC3,SIMBA,신항 2부두 2선석,<NA>,2022-04-30 23:00,2022-05-02 08:00,<NA>,<NA>,True,True
5388,V7HX8,HEUNG-A JANICE,신항 4부두 3선석,<NA>,2022-05-12 02:00,2022-05-12 20:00,<NA>,<NA>,True,True
5828,BOTT,LING YUN HE,신항 5부두 1선석,<NA>,2022-05-22 12:00,2022-05-23 12:00,<NA>,<NA>,True,True
6405,9HA5531,CMA CGM OSIRIS,신항 5부두 2선석,<NA>,2022-06-05 09:00,2022-06-06 09:00,<NA>,<NA>,True,True
7143,VRNR5,OOCL UTAH,신항 5부두 1선석,<NA>,2022-06-23 03:00,2022-06-23 20:00,<NA>,<NA>,True,True
